# AiXbio — Notebook 0: Setup & Data Download
Run on **Lambda Cloud** (GPU instance).

In [ ]:
# ── Install all dependencies ──────────────────────────────────
import subprocess, sys, os
from pathlib import Path

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=False)

pip('transformers', 'torch', 'accelerate',
    'biopython', 'requests', 'tqdm',
    'numpy', 'pandas', 'scikit-learn',
    'matplotlib', 'seaborn', 'umap-learn',
    'devinterp', 'huggingface_hub')

# InterPLM — git clone + pip install -e . (not on PyPI)
subprocess.run(
    'git clone --quiet https://github.com/ElanaPearl/interPLM.git /tmp/interPLM 2>/dev/null || true',
    shell=True
)
pip('-e', '/tmp/interPLM')

# ── MMseqs2 via wget (fastest on Lambda Cloud) ────────────────
# If you already ran the wget command manually in terminal, skip this cell.
mmseqs_bin = Path('mmseqs/bin/mmseqs')
if not mmseqs_bin.exists():
    print('Downloading MMseqs2...')
    subprocess.run(
        'wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz '
        '&& tar xfz mmseqs-linux-avx2.tar.gz',
        shell=True, check=True
    )

# ✓ Inject mmseqs/bin into PATH so all subprocess calls can find it
mmseqs_bin_dir = str(Path('mmseqs/bin').resolve())
if mmseqs_bin_dir not in os.environ.get('PATH', ''):
    os.environ['PATH'] = mmseqs_bin_dir + ':' + os.environ.get('PATH', '')
    print(f'Added to PATH: {mmseqs_bin_dir}')

# Verify
check = subprocess.run('mmseqs version', shell=True, capture_output=True, text=True)
if check.returncode == 0:
    print(f'MMseqs2 ready: v{check.stdout.strip()}')
else:
    print('MMseqs2 not found — Python fallback will be used for clustering')

# BLAST+
subprocess.run(
    'sudo apt-get install -y ncbi-blast+ 2>/dev/null || echo "BLAST+ unavailable — pairwise identity proxy will be used"',
    shell=True
)

# ProteinMPNN
subprocess.run(
    'git clone --quiet https://github.com/dauparas/ProteinMPNN /tmp/ProteinMPNN 2>/dev/null || true',
    shell=True
)
print('Setup complete.')

In [ ]:
# ── Imports & global config ───────────────────────────────────
import os, json, time, warnings, re
from pathlib import Path
from io import StringIO
import numpy as np
import requests
from tqdm.auto import tqdm
import torch
from Bio import SeqIO
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Re-ensure mmseqs is on PATH if kernel was restarted
mmseqs_bin_dir = str(Path('mmseqs/bin').resolve())
if mmseqs_bin_dir not in os.environ.get('PATH', ''):
    os.environ['PATH'] = mmseqs_bin_dir + ':' + os.environ.get('PATH', '')

LAYERS = [1, 9, 18, 24, 30, 33]

for d in ['data/structures', 'embeddings', 'redesign/outputs',
           'probes', 'sae', 'figures', 'results']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Directories ready.')

## 1. Download UniProt Sequences

In [ ]:
def download_uniprot(query: str, out_fasta: str, max_seqs: int = 10000):
    base = 'https://rest.uniprot.org/uniprotkb/search'
    params = {'query': query, 'format': 'fasta', 'size': 500}
    records, cursor = [], None
    pbar = tqdm(desc=Path(out_fasta).name, unit=' seq')

    while True:
        if cursor:
            params['cursor'] = cursor
        r = requests.get(base, params=params, timeout=60)
        r.raise_for_status()
        batch = list(SeqIO.parse(StringIO(r.text), 'fasta'))
        records.extend(batch)
        pbar.update(len(batch))
        link = r.headers.get('Link', '')
        if 'rel="next"' not in link or len(records) >= max_seqs:
            break
        m = re.search(r'cursor=([^&>]+)', link)
        cursor = m.group(1) if m else None
        if not cursor:
            break
        time.sleep(0.5)

    pbar.close()
    out = records[:max_seqs]
    SeqIO.write(out, out_fasta, 'fasta')
    print(f'  Saved {len(out)} → {out_fasta}')
    return out


toxin_records = download_uniprot(
    query='keyword:KW-0800 AND reviewed:true AND length:[10 TO 2000]',
    out_fasta='data/toxins_raw.fasta'
)
control_records = download_uniprot(
    query=('reviewed:true AND length:[10 TO 2000]'
           ' NOT keyword:KW-0800 NOT keyword:KW-0843'
           ' NOT go:"GO:0009405"'),
    out_fasta='data/controls_raw.fasta',
    max_seqs=12000
)

In [ ]:
# ── Length-match controls to toxin distribution ───────────────
toxins   = list(SeqIO.parse('data/toxins_raw.fasta', 'fasta'))
controls = list(SeqIO.parse('data/controls_raw.fasta', 'fasta'))

tox_lengths  = np.array([len(r.seq) for r in toxins])
ctrl_lengths = np.array([len(r.seq) for r in controls])

rng = np.random.RandomState(42)
bins = np.percentile(tox_lengths, np.linspace(0, 100, 11))
selected_ctrl = []
n_per_bin = len(toxins) // 10 + 1

for lo, hi in zip(bins[:-1], bins[1:]):
    cands = [r for r, l in zip(controls, ctrl_lengths) if lo <= l <= hi]
    k = min(n_per_bin, len(cands))
    if k > 0:
        # FIX: choose integer indices — rng.choice on SeqRecord objects
        # raises ValueError (inhomogeneous array). Index the list manually.
        idx = rng.choice(len(cands), k, replace=False)
        selected_ctrl.extend([cands[i] for i in idx])

selected_ctrl = selected_ctrl[:len(toxins)]
SeqIO.write(selected_ctrl, 'data/controls_matched.fasta', 'fasta')
print(f'Toxins: {len(toxins)}, Controls (length-matched): {len(selected_ctrl)}')

## 2. Sequence Clustering (MMseqs2 at 30% identity)

In [ ]:
import subprocess
from Bio import pairwise2

def run_mmseqs2(fasta_in: str, fasta_out: str, identity: float = 0.30):
    """Cluster at given identity. PATH must include mmseqs/bin (set in cell 1)."""
    tmp = '/tmp/mmseqs_tmp'
    Path(tmp).mkdir(exist_ok=True)
    prefix = Path(fasta_out).stem
    db   = f'/tmp/{prefix}_db'
    clst = f'/tmp/{prefix}_clst'

    cmds = [
        f'mmseqs createdb {fasta_in} {db}',
        f'mmseqs cluster {db} {clst} {tmp} --min-seq-id {identity} -c 0.8 --cov-mode 0 -v 0',
        f'mmseqs createsubdb {clst} {db} {clst}_rep',
        f'mmseqs convert2fasta {clst}_rep {fasta_out}',
    ]
    for cmd in cmds:
        # Pass os.environ so mmseqs/bin is on PATH inside subprocess
        res = subprocess.run(cmd, shell=True, capture_output=True, env=os.environ)
        if res.returncode != 0:
            raise RuntimeError(f'Command failed: {cmd}\n{res.stderr.decode()}')

    recs = list(SeqIO.parse(fasta_out, 'fasta'))
    print(f'  {Path(fasta_in).name} → {len(recs)} reps at {identity*100:.0f}% identity')
    return recs


def greedy_cluster(records, threshold=0.30, max_check=150):
    """Pure-Python greedy clustering fallback."""
    reps = []
    for rec in tqdm(records, desc='Greedy cluster'):
        seq = str(rec.seq)
        covered = False
        for r in reps:
            rseq = str(r.seq)
            if abs(len(seq) - len(rseq)) / max(len(seq), len(rseq), 1) > 0.7:
                continue
            q, s = seq[:max_check], rseq[:max_check]
            score = pairwise2.align.globalxx(q, s, score_only=True)
            if score / max(len(q), len(s)) >= threshold:
                covered = True
                break
        if not covered:
            reps.append(rec)
    return reps


try:
    print('Running MMseqs2...')
    tox_reps  = run_mmseqs2('data/toxins_raw.fasta',       'data/toxins_clustered.fasta')
    ctrl_reps = run_mmseqs2('data/controls_matched.fasta', 'data/controls_clustered.fasta')
except Exception as e:
    print(f'MMseqs2 failed ({e})\nUsing Python greedy fallback...')
    tox_reps  = greedy_cluster(list(SeqIO.parse('data/toxins_raw.fasta', 'fasta')))
    ctrl_reps = greedy_cluster(list(SeqIO.parse('data/controls_matched.fasta', 'fasta')))
    SeqIO.write(tox_reps,  'data/toxins_clustered.fasta',  'fasta')
    SeqIO.write(ctrl_reps, 'data/controls_clustered.fasta', 'fasta')

print(f'Representatives — Toxins: {len(tox_reps)}, Controls: {len(ctrl_reps)}')

## 3. Download AlphaFold Structures (100 toxin reps)

In [ ]:
def get_uniprot_id(record):
    m = re.search(r'[sptr]+\|([A-Z0-9]+)\|', record.id)
    return m.group(1) if m else record.id.split('|')[0].split()[0]


def download_alphafold_pdb(uid: str, out_path: str) -> bool:
    if Path(out_path).exists():
        return True
    url = f'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            Path(out_path).write_text(r.text)
            return True
    except Exception:
        pass
    return False


tox_reps = list(SeqIO.parse('data/toxins_clustered.fasta', 'fasta'))
struct_dir = Path('data/structures')
downloaded, failed = [], []

for rec in tqdm(tox_reps[:100], desc='AlphaFold'):
    uid = get_uniprot_id(rec)
    out = str(struct_dir / f'{uid}.pdb')
    if download_alphafold_pdb(uid, out):
        downloaded.append({'uniprot_id': uid, 'pdb_path': out, 'sequence': str(rec.seq)})
    else:
        failed.append(uid)
    time.sleep(0.2)

print(f'Downloaded: {len(downloaded)}, Not in AlphaFold DB: {len(failed)}')
with open('data/structures_manifest.json', 'w') as f:
    json.dump(downloaded, f, indent=2)
print('Manifest → data/structures_manifest.json')